# 01 — Feature Engineering

Build the gold feature table from the raw flotation dataset:

1. **Rolling statistics** (mean / std / min / max) over 10 min, 1 h, 3 h windows.
2. **Lag features** at 10 min, 1 h, 3 h (process delay reflection).
3. **Calendar features**: hour, day-of-week, weekend flag.

Result: a gold parquet table at `data/processed/gold.parquet`, ready for modeling notebooks.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from frothiq.data.loader import SENSOR_COLS, FEED_COLS, TARGET_COLS, load_flotation
from frothiq.features.pipeline import FeatureConfig, build_features, save_gold, list_feature_cols

## 1. Load raw data

In [2]:
data = load_flotation()
print(f'Loaded {len(data.df):,} rows × {len(data.df.columns)} columns')

Loaded 737,453 rows × 24 columns


## 2. Build features

Default windows: 30 rows (10 min), 180 rows (1 h), 540 rows (3 h) at 20-second sampling.

In [3]:
config = FeatureConfig(
    rolling_windows=(30, 180, 540),
    lag_steps=(30, 180, 540),
    add_calendar=True,
    include_feed=True,
)

t0 = time.time()
gold = build_features(data.df, sensor_cols=data.sensor_cols, feed_cols=data.feed_cols, config=config)
elapsed = time.time() - t0
print(f'Built {gold.shape[1]} columns over {len(gold):,} rows in {elapsed:.1f}s')
print(f'Memory footprint: {gold.memory_usage(deep=True).sum() / 1e6:.1f} MB')

Built 312 columns over 737,453 rows in 7.1s
Memory footprint: 1825.2 MB


## 3. Sanity-check correlations with target

In [4]:
feature_cols = list_feature_cols(gold, target_cols=TARGET_COLS)
print(f'Total feature columns: {len(feature_cols)}')

for target in TARGET_COLS:
    corrs = gold[feature_cols].corrwith(gold[target]).abs().sort_values(ascending=False)
    print(f'\nTop 10 features by |corr| with {target}:')
    print(corrs.head(10).round(3).to_string())

Total feature columns: 309

Top 10 features by |corr| with pct_iron_concentrate:
ore_pulp_ph_mean_540    0.222
ore_pulp_ph_max_540     0.219
ore_pulp_ph_lag_540     0.207
ore_pulp_ph_max_180     0.204
ore_pulp_ph_min_540     0.202
ore_pulp_ph_mean_180    0.202
ore_pulp_ph_lag_180     0.201
ore_pulp_ph_min_180     0.198
ore_pulp_ph_max_30      0.195
ore_pulp_ph_lag_30      0.195

Top 10 features by |corr| with pct_silica_concentrate:
flot_col_01_air_flow_mean_540    0.225
flot_col_03_air_flow_mean_540    0.223
flot_col_01_air_flow_mean_180    0.222
flot_col_01_air_flow_lag_540     0.221
flot_col_03_air_flow_mean_180    0.221
flot_col_01_air_flow_lag_180     0.221
flot_col_03_air_flow_min_30      0.220
flot_col_03_air_flow_lag_180     0.220
flot_col_03_air_flow_min_180     0.220
flot_col_01_air_flow_mean_30     0.220


## 4. Persist gold for downstream notebooks

In [5]:
out_path = save_gold(gold, ROOT / 'data' / 'processed' / 'gold.parquet')
print(f'Saved {len(gold):,} rows × {gold.shape[1]} cols to {out_path}')

Saved 737,453 rows × 312 cols to C:\Users\jona2\frothiq\data\processed\gold.parquet


---

## Next: `02_baseline_lightgbm.ipynb`

Train LightGBM regressors for `pct_iron_concentrate` and `pct_silica_concentrate`. Track in MLflow. Compare against the naive baseline from notebook 00.